In [4]:
import os
import time
import traceback
import numpy as np
import torch

import matplotlib

# 非交互式后端：保存图片但不弹窗
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from ase import Atoms
from ase.calculators.calculator import (
    Calculator,
    all_changes,
)


# ============================================================
# 导入包含 FENEB 的自定义 neb.py
#
# 优先导入当前目录下的 neb.py。
# ============================================================
try:
    from neb import NEB, NEBOptimizer

    FENEB_IMPORT_SOURCE = "local neb.py"

except ImportError:
    try:
        from ase.mep.neb import (
            NEB,
            NEBOptimizer,
        )

        FENEB_IMPORT_SOURCE = "ase.mep.neb"

    except ImportError:
        from ase.neb import (
            NEB,
            NEBOptimizer,
        )

        FENEB_IMPORT_SOURCE = "ase.neb"


# ============================================================
# Alanine restrained-MD oracle 和绘图函数
#
# IVM_shell.py 最后必须有：
#
# if __name__ == "__main__":
#     main()
# ============================================================
import IVM_shell as ala_ivm


M_gradient = ala_ivm.M_gradient
fig_save = ala_ivm.fig_save

if hasattr(ala_ivm, "GROMACS_QUIET"):
    ala_ivm.GROMACS_QUIET = True


# ============================================================
# 基本设置
# ============================================================
device = torch.device("cpu")
dtype = torch.float64

seed = 41

np.random.seed(seed)
torch.manual_seed(seed)


# ============================================================
# 与 IVM / ODE-NEB 完全相同的端点
# ============================================================
C1 = np.array(
    [-2.60866606, 2.77661136],
    dtype=float,
)

C2 = np.array(
    [-1.3304716, 0.99118795],
    dtype=float,
)


# ============================================================
# Images 数量
#
# N 是内部 images 数。
# 总 images 数为 N + 2。
# ============================================================
N = 20
N_TOTAL_IMAGES = N + 2


# ============================================================
# 输出路径
# ============================================================
base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)


# ============================================================
# Restrained-MD 参数
# ============================================================
DISCARD_FRAC = 0.05
TMIN = None
TMAX = None


# ============================================================
# FENEB grid
# ============================================================
GRID_K = [
    1.0,
]

GRID_ALPHA = [
    1e-4,
    1e-5,
    1e-6,
    1e-7
]


# ============================================================
# 最大 count / outer step
#
# 初始状态：
#     count = 0
#
# 完成 200 次更新后：
#     count = 200
# ============================================================
MAX_FENEB_STEPS = 200


# ============================================================
# FENEB physical-force 停止条件
#
# 如果在 count=200 之前达到，会提前停止。
#
# 如果希望强制跑满 200，可设为非常小：
#
# FENEB_FMAX_TOL = 1e-12
# ============================================================
FENEB_FMAX_TOL = 0.01


# ============================================================
# 重参数化参数
#
# 重要：
#
# physical update 使用 grid alpha：
#
#     X += alpha * F
#
# reparameterization 固定使用：
#
#     X += REPARAM_ALPHA * F_reparam
#
# 两者不再共用 alpha。
# ============================================================
REPARAM_ALPHA = 1e-3

# 重参数化停止条件：
#
#     max_norm / k <= REPARAM_TOL
#
REPARAM_TOL = 1e-3

# 每个 outer step 最多进行多少次重参数化子步
#
# 如果达到此上限仍未满足 REPARAM_TOL：
#
#     只警告并继续
#     不报错
#     不停止整个实验
#     不写 FAILED
#
MAX_REPARAM_SUBSTEPS = 100000

# 重参数化迭代打印间隔
REPARAM_PRINT_EVERY = 5000


# ============================================================
# 是否重新运行
# ============================================================
FORCE_RERUN = True


# ============================================================
# 保存与绘图
# ============================================================
SAVE_EVERY_STATE = True

SAVE_FIGS = True
SAVE_FIG_EVERY = 10

xlim = (-np.pi, 0.0)
ylim = (0.0, np.pi)


# ============================================================
# 全局 restrained-MD 调用次数
# ============================================================
GLOBAL_IMAGE_MD_CALLS = 0


# ============================================================
# CV <-> ASE Atoms
#
# 两个原子的 x 坐标保存二维 CV：
#
# atom 0, x = phi
# atom 1, x = psi
# ============================================================
def atoms_to_q(atoms):
    positions = atoms.get_positions()

    return np.array(
        [
            positions[0, 0],
            positions[1, 0],
        ],
        dtype=float,
    )


def make_cv_atoms(
    q,
    calculator=None,
):
    q = np.asarray(
        q,
        dtype=float,
    ).reshape(2,)

    positions = np.zeros(
        (2, 3),
        dtype=float,
    )

    positions[0, 0] = q[0]
    positions[1, 0] = q[1]

    atoms = Atoms(
        "H2",
        positions=positions,
    )

    atoms.pbc = False

    if calculator is not None:
        atoms.calc = calculator

    return atoms


# ============================================================
# 固定端点 calculator
# ============================================================
class ZeroEndpointCalculator(Calculator):
    implemented_properties = [
        "energy",
        "forces",
    ]

    def calculate(
        self,
        atoms=None,
        properties=("energy",),
        system_changes=all_changes,
    ):
        super().calculate(
            atoms,
            properties,
            system_changes,
        )

        self.results["energy"] = 0.0

        self.results["forces"] = np.zeros(
            (len(atoms), 3),
            dtype=float,
        )


# ============================================================
# FENEB 内部 image calculator
#
# restrained-MD oracle 由 feneb_gradient_md_use() 统一调用。
# ============================================================
class AlanineFENEBImageCalculator(Calculator):
    implemented_properties = [
        "energy",
        "forces",
    ]

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.md_calls = 0

        self.last_q = None
        self.last_M = None
        self.last_F = None
        self.last_Mg = None

    def calculate(
        self,
        atoms=None,
        properties=("energy",),
        system_changes=all_changes,
    ):
        super().calculate(
            atoms,
            properties,
            system_changes,
        )

        self.results["energy"] = 0.0

        self.results["forces"] = np.zeros(
            (len(atoms), 3),
            dtype=float,
        )


# ============================================================
# 直线初始路径
# ============================================================
def get_linear_initial_path(
    q0,
    q1,
    n_internal,
):
    t = np.linspace(
        0.0,
        1.0,
        n_internal + 2,
    )[1:-1]

    return (
        (1.0 - t[:, None]) * q0[None, :]
        + t[:, None] * q1[None, :]
    )


# ============================================================
# 每个实验重新构造 images
# ============================================================
def build_feneb_images():
    initial_internal = get_linear_initial_path(
        C1,
        C2,
        N,
    )

    images = []

    images.append(
        make_cv_atoms(
            C1,
            calculator=ZeroEndpointCalculator(),
        )
    )

    for q in initial_internal:
        images.append(
            make_cv_atoms(
                q,
                calculator=(
                    AlanineFENEBImageCalculator()
                ),
            )
        )

    images.append(
        make_cv_atoms(
            C2,
            calculator=ZeroEndpointCalculator(),
        )
    )

    return images


# ============================================================
# 提取完整二维路径
# ============================================================
def images_to_path(images):
    return np.asarray(
        [
            atoms_to_q(image)
            for image in images
        ],
        dtype=float,
    )


# ============================================================
# Restrained-MD oracle adapter
#
# 自定义 neb.py 中：
#
#     forces = -self.gradient_md_use(self.images)
#
# 所以本函数返回 +M grad F。
# ============================================================
def feneb_gradient_md_use(images):
    global GLOBAL_IMAGE_MD_CALLS

    n_internal = len(images) - 2

    output = np.zeros(
        (n_internal, 2, 3),
        dtype=float,
    )

    for internal_index, image in enumerate(
        images[1:-1]
    ):
        calculator = image.calc

        if not isinstance(
            calculator,
            AlanineFENEBImageCalculator,
        ):
            raise TypeError(
                f"Internal image "
                f"{internal_index + 1} "
                "does not use "
                "AlanineFENEBImageCalculator."
            )

        q = atoms_to_q(
            image
        )

        q_tensor = torch.tensor(
            q.reshape(1, 2),
            dtype=dtype,
            device=device,
        )

        M, F_orig = M_gradient(
            q_tensor,
            discard_frac=DISCARD_FRAC,
            tmin=TMIN,
            tmax=TMAX,
        )

        M = np.asarray(
            M,
            dtype=float,
        ).reshape(2, 2)

        F_orig = np.asarray(
            F_orig,
            dtype=float,
        ).reshape(2,)

        if not np.all(np.isfinite(M)):
            raise FloatingPointError(
                f"Non-finite M at q={q}:\n"
                f"{M}"
            )

        if not np.all(np.isfinite(F_orig)):
            raise FloatingPointError(
                f"Non-finite F_orig at q={q}: "
                f"{F_orig}"
            )

        # kJ/mol -> kcal/mol
        F = F_orig / 4.184

        # Riemannian gradient
        Mg = M @ F

        if not np.all(np.isfinite(Mg)):
            raise FloatingPointError(
                f"Non-finite M @ F at q={q}:\n"
                f"M = {M}\n"
                f"F = {F}\n"
                f"M @ F = {Mg}"
            )

        output[
            internal_index,
            0,
            0,
        ] = Mg[0]

        output[
            internal_index,
            1,
            0,
        ] = Mg[1]

        calculator.last_q = q.copy()
        calculator.last_M = M.copy()
        calculator.last_F = F.copy()
        calculator.last_Mg = Mg.copy()

        calculator.md_calls += 1
        GLOBAL_IMAGE_MD_CALLS += 1

        image.info["M"] = M.copy()
        image.info["F"] = F.copy()
        image.info["Mg"] = Mg.copy()

    return output


# ============================================================
# 收集当前 oracle 数据
# ============================================================
def collect_internal_oracle_data(images):
    M_list = []
    F_list = []
    Mg_list = []

    for image_index, image in enumerate(
        images[1:-1],
        start=1,
    ):
        calculator = image.calc

        if not isinstance(
            calculator,
            AlanineFENEBImageCalculator,
        ):
            raise TypeError(
                f"Internal image {image_index} "
                "does not use "
                "AlanineFENEBImageCalculator."
            )

        if (
            calculator.last_q is None
            or calculator.last_M is None
            or calculator.last_F is None
            or calculator.last_Mg is None
        ):
            raise RuntimeError(
                f"No oracle data available for "
                f"internal image {image_index}."
            )

        current_q = atoms_to_q(
            image
        )

        if not np.allclose(
            current_q,
            calculator.last_q,
            rtol=0.0,
            atol=1e-10,
        ):
            raise RuntimeError(
                f"Stale oracle data for image "
                f"{image_index}:\n"
                f"current q = {current_q}\n"
                f"cached q  = "
                f"{calculator.last_q}"
            )

        M_list.append(
            calculator.last_M.copy()
        )

        F_list.append(
            calculator.last_F.copy()
        )

        Mg_list.append(
            calculator.last_Mg.copy()
        )

    return (
        np.stack(
            M_list,
            axis=0,
        ),
        np.stack(
            F_list,
            axis=0,
        ),
        np.stack(
            Mg_list,
            axis=0,
        ),
    )


# ============================================================
# Oracle 调用次数
# ============================================================
def get_internal_image_md_calls(images):
    calls = []

    for image_index, image in enumerate(
        images[1:-1],
        start=1,
    ):
        calculator = image.calc

        if not isinstance(
            calculator,
            AlanineFENEBImageCalculator,
        ):
            raise TypeError(
                f"Internal image {image_index} "
                "does not use "
                "AlanineFENEBImageCalculator."
            )

        calls.append(
            int(calculator.md_calls)
        )

    calls = np.asarray(
        calls,
        dtype=int,
    )

    if calls.size == 0:
        raise RuntimeError(
            "No internal images found."
        )

    return (
        calls,
        int(calls[0]),
    )


# ============================================================
# 中心差分 tangent
#
# FENEB 的 spring1.t + spring2.t 等价于：
#
# path[i+1] - path[i-1]
# ============================================================
def central_difference_tangent(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    return (
        full_path[2:]
        - full_path[:-2]
    )


# ============================================================
# G-正交投影
#
# G = M^{-1}
# ============================================================
def compute_G_projectors(
    tangent,
    M_all,
):
    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    n_internal = tangent.shape[0]

    if tangent.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"Unexpected tangent shape: "
            f"{tangent.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    G_all = np.linalg.inv(
        M_all
    )

    P_parallel_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    P_perp_all = np.zeros(
        (n_internal, 2, 2),
        dtype=float,
    )

    identity = np.eye(
        2,
        dtype=float,
    )

    for i in range(n_internal):
        v = tangent[i]
        G_i = G_all[i]

        norm_G_sq = float(
            v @ G_i @ v
        )

        if norm_G_sq <= 1e-30:
            raise FloatingPointError(
                f"Near-zero G tangent norm "
                f"at image {i}: {norm_G_sq}"
            )

        tau_G = (
            v
            / np.sqrt(norm_G_sq)
        )

        G_tau_G = G_i @ tau_G

        denom = float(
            tau_G @ G_tau_G
        )

        P_parallel = (
            np.outer(
                tau_G,
                G_tau_G,
            )
            / max(denom, 1e-30)
        )

        P_parallel_all[i] = (
            P_parallel
        )

        P_perp_all[i] = (
            identity
            - P_parallel
        )

    return (
        G_all,
        P_parallel_all,
        P_perp_all,
    )


# ============================================================
# Euclidean normal projection
# ============================================================
def compute_euclidean_perp(
    vectors,
    tangent,
):
    vectors = np.asarray(
        vectors,
        dtype=float,
    )

    tangent = np.asarray(
        tangent,
        dtype=float,
    )

    tangent_norm = np.linalg.norm(
        tangent,
        axis=1,
        keepdims=True,
    )

    tau = tangent / (
        tangent_norm + 1e-16
    )

    inner = np.sum(
        tau * vectors,
        axis=1,
        keepdims=True,
    )

    return (
        vectors
        - inner * tau
    )


# ============================================================
# 路径积分重建自由能
# ============================================================
def recover_free_energy_profile(
    full_path,
    F_internal,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    F_internal = np.asarray(
        F_internal,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if F_internal.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"F shape {F_internal.shape} "
            f"is inconsistent with path "
            f"shape {full_path.shape}"
        )

    segments = (
        full_path[1:]
        - full_path[:-1]
    )

    segment_gradients = np.zeros(
        (n_internal + 1, 2),
        dtype=float,
    )

    segment_gradients[0] = (
        F_internal[0]
    )

    if n_internal > 1:
        segment_gradients[
            1:n_internal
        ] = (
            0.5
            * (
                F_internal[:-1]
                + F_internal[1:]
            )
        )

    segment_gradients[
        n_internal
    ] = F_internal[-1]

    increments = np.einsum(
        "ni,ni->n",
        segment_gradients,
        segments,
    )

    profile = np.zeros(
        n_internal + 2,
        dtype=float,
    )

    profile[1:] = np.cumsum(
        increments
    )

    profile -= profile[0]

    barrier = float(
        np.max(profile)
        - profile[0]
    )

    max_index = int(
        np.argmax(profile)
    )

    final_delta_F = float(
        profile[-1]
        - profile[0]
    )

    return {
        "profile": profile,
        "barrier": barrier,
        "max_index": max_index,
        "final_delta_F": final_delta_F,
        "increments": increments,
    }


# ============================================================
# Arclength / equi-speed residual
# ============================================================
def compute_arclength_residual(
    full_path,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    if (
        full_path.ndim != 2
        or full_path.shape[1] != 2
    ):
        raise ValueError(
            f"full_path must have shape "
            f"(n,2), got {full_path.shape}"
        )

    n_full = full_path.shape[0]

    if n_full < 3:
        raise ValueError(
            "Path must contain at least "
            "one internal image."
        )

    ds = 1.0 / (
        n_full - 1
    )

    phi_prime = (
        full_path[2:]
        - full_path[:-2]
    ) / (2.0 * ds)

    phi_second = (
        full_path[2:]
        - 2.0 * full_path[1:-1]
        + full_path[:-2]
    ) / (ds ** 2)

    inner = np.einsum(
        "ni,ni->n",
        phi_prime,
        phi_second,
    )

    speed = np.linalg.norm(
        phi_prime,
        axis=1,
    )

    integrand = (
        inner ** 2
        * speed
    )

    I_arc = float(
        np.sum(integrand)
        * ds
    )

    return {
        "I_arc": I_arc,
        "phi_prime": phi_prime,
        "phi_second": phi_second,
        "inner": inner,
        "speed": speed,
        "integrand": integrand,
        "ds": ds,
    }


# ============================================================
# 四个比较指标
# ============================================================
def compute_all_path_metrics(
    full_path,
    M_all,
    F_all,
):
    full_path = np.asarray(
        full_path,
        dtype=float,
    )

    M_all = np.asarray(
        M_all,
        dtype=float,
    )

    F_all = np.asarray(
        F_all,
        dtype=float,
    )

    n_internal = (
        full_path.shape[0]
        - 2
    )

    if full_path.shape != (
        n_internal + 2,
        2,
    ):
        raise ValueError(
            f"Unexpected path shape: "
            f"{full_path.shape}"
        )

    if M_all.shape != (
        n_internal,
        2,
        2,
    ):
        raise ValueError(
            f"Unexpected M shape: "
            f"{M_all.shape}"
        )

    if F_all.shape != (
        n_internal,
        2,
    ):
        raise ValueError(
            f"Unexpected F shape: "
            f"{F_all.shape}"
        )

    tangent = central_difference_tangent(
        full_path
    )

    (
        G_all,
        P_parallel_G_all,
        P_perp_G_all,
    ) = compute_G_projectors(
        tangent,
        M_all,
    )

    Mg_all = np.einsum(
        "nij,nj->ni",
        M_all,
        F_all,
    )

    perpG_Mg = np.einsum(
        "nij,nj->ni",
        P_perp_G_all,
        Mg_all,
    )

    perpE_Mg = compute_euclidean_perp(
        Mg_all,
        tangent,
    )

    path_internal = (
        full_path[1:-1]
    )

    seg_minus = (
        path_internal
        - full_path[:-2]
    )

    seg_plus = (
        full_path[2:]
        - path_internal
    )

    # G-arclength weights
    seg_minus_G_sq = np.einsum(
        "ni,nij,nj->n",
        seg_minus,
        G_all,
        seg_minus,
    )

    seg_plus_G_sq = np.einsum(
        "ni,nij,nj->n",
        seg_plus,
        G_all,
        seg_plus,
    )

    seg_minus_G = np.sqrt(
        np.maximum(
            seg_minus_G_sq,
            0.0,
        )
    )

    seg_plus_G = np.sqrt(
        np.maximum(
            seg_plus_G_sq,
            0.0,
        )
    )

    node_weights_G = 0.5 * (
        seg_minus_G
        + seg_plus_G
    )

    # Euclidean arclength weights
    seg_minus_E = np.linalg.norm(
        seg_minus,
        axis=1,
    )

    seg_plus_E = np.linalg.norm(
        seg_plus,
        axis=1,
    )

    node_weights_E = 0.5 * (
        seg_minus_E
        + seg_plus_E
    )

    # G residual
    perpG_norm_G_sq = np.einsum(
        "ni,nij,nj->n",
        perpG_Mg,
        G_all,
        perpG_Mg,
    )

    perpG_norm_G_sq = np.maximum(
        perpG_norm_G_sq,
        0.0,
    )

    I_perp_G = float(
        np.sum(
            perpG_norm_G_sq
            * node_weights_G
        )
    )

    # Euclidean residual
    perpG_norm_E_sq = np.sum(
        perpG_Mg ** 2,
        axis=1,
    )

    I_perp_E = float(
        np.sum(
            perpG_norm_E_sq
            * node_weights_E
        )
    )

    fmax_Gperp_Euclidean = float(
        np.max(
            np.sqrt(
                perpG_norm_E_sq
            )
        )
    )

    fmax_FENEB_physical = float(
        np.max(
            np.linalg.norm(
                perpE_Mg,
                axis=1,
            )
        )
    )

    arc_result = compute_arclength_residual(
        full_path
    )

    free_energy = recover_free_energy_profile(
        full_path,
        F_all,
    )

    return {
        "G_all": G_all,

        "P_parallel_G_all": (
            P_parallel_G_all
        ),

        "P_perp_G_all": (
            P_perp_G_all
        ),

        "Mg_all": Mg_all,
        "perpG_Mg": perpG_Mg,
        "perpE_Mg": perpE_Mg,

        "node_weights_G": (
            node_weights_G
        ),

        "node_weights_E": (
            node_weights_E
        ),

        "I_perp_G": I_perp_G,
        "I_perp_E": I_perp_E,

        "I_arc": (
            arc_result["I_arc"]
        ),

        "arc_integrand": (
            arc_result["integrand"]
        ),

        "arc_speed": (
            arc_result["speed"]
        ),

        "arc_inner": (
            arc_result["inner"]
        ),

        "Delta_F_barrier": (
            free_energy["barrier"]
        ),

        "Delta_F_final": (
            free_energy["final_delta_F"]
        ),

        "free_energy_profile": (
            free_energy["profile"]
        ),

        "fmax_Gperp_Euclidean": (
            fmax_Gperp_Euclidean
        ),

        "fmax_FENEB_physical": (
            fmax_FENEB_physical
        ),
    }


# ============================================================
# 文件名工具
# ============================================================
def float_tag(value):
    return f"{float(value):g}"


def feneb_result_paths(
    k_spring,
    alpha,
):
    k_tag = float_tag(
        k_spring
    )

    alpha_tag = float_tag(
        alpha
    )

    casenum = (
        f"MFEP_FENEB_N{N}"
        f"_k{k_tag}"
        f"_alpha{alpha_tag}"
        f"_reparamAlpha{REPARAM_ALPHA:g}"
        f"_reparamTol{REPARAM_TOL:g}"
        f"_fmax{FENEB_FMAX_TOL:g}"
        f"_steps{MAX_FENEB_STEPS}"
        f"_centralTangent"
        f"_straightInit"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    figure_path = os.path.join(
        folder_path,
        "figures",
    )

    result_file = os.path.join(
        folder_path,
        f"alanine_{casenum}.npz",
    )

    done_file = os.path.join(
        folder_path,
        "DONE",
    )

    failed_file = os.path.join(
        folder_path,
        "FAILED",
    )

    return (
        casenum,
        folder_path,
        figure_path,
        result_file,
        done_file,
        failed_file,
    )


# ============================================================
# 原子化保存 npz
# ============================================================
def atomic_savez(
    result_file,
    **kwargs,
):
    temporary_file = (
        result_file
        + ".tmp.npz"
    )

    np.savez(
        temporary_file,
        **kwargs,
    )

    os.replace(
        temporary_file,
        result_file,
    )


# ============================================================
# FENEB optimizer
#
# 关键变化：
#
# physical:
#     X += self.alpha * F
#
# reparameterization:
#     X += self.reparam_alpha * F_reparam
#
# 达到最大 reparam substeps 后：
#     warning + continue
# ============================================================
class RecordingFENEBOptimizer(
    NEBOptimizer
):
    def __init__(
        self,
        *args,
        state_recorder=None,
        reparam_alpha=REPARAM_ALPHA,
        reparam_tol=REPARAM_TOL,
        max_reparam_substeps=(
            MAX_REPARAM_SUBSTEPS
        ),
        reparam_print_every=(
            REPARAM_PRINT_EVERY
        ),
        **kwargs,
    ):
        super().__init__(
            *args,
            **kwargs,
        )

        self.state_recorder = (
            state_recorder
        )

        self.reparam_alpha = float(
            reparam_alpha
        )

        self.reparam_tol = float(
            reparam_tol
        )

        self.max_reparam_substeps = int(
            max_reparam_substeps
        )

        self.reparam_print_every = int(
            reparam_print_every
        )

        self.last_reparam_substeps = 0
        self.total_reparam_substeps = 0

        self.last_reparam_converged = True
        self.last_reparam_max_norm = 0.0
        self.last_reparam_ratio = 0.0

        self.reparam_limit_hits = 0

    def record_current_state(
        self,
        physical_forces,
    ):
        if self.state_recorder is None:
            return

        self.state_recorder(
            optimizer=self,
            physical_forces=physical_forces,
        )

    def run_FENEB(self, fmax):
        X = (
            self.neb
            .get_positions()
            .reshape(-1)
        )

        for _ in range(
            self.max_steps
        ):
            # --------------------------------------------
            # 当前路径上的 restrained-MD physical force
            # --------------------------------------------
            F = self.normalized_force_function(
                X
            )

            # 保存当前路径，保证 path 与 M/F 对应
            self.record_current_state(
                physical_forces=F,
            )

            current_residual = float(
                self.neb.get_residual()
            )

            if current_residual <= fmax:
                return True

            # --------------------------------------------
            # Physical update
            #
            # 使用当前 grid 中正在调节的 alpha
            # --------------------------------------------
            X = (
                X
                + self.alpha * F
            )

            # --------------------------------------------
            # Reparameterization
            #
            # 固定使用 REPARAM_ALPHA，不使用 grid alpha
            # --------------------------------------------
            (
                F_reparam,
                max_norm,
            ) = self.reparam_force_function(
                X
            )

            if not np.isfinite(max_norm):
                raise FloatingPointError(
                    "Non-finite FENEB "
                    "reparameterization max_norm: "
                    f"{max_norm}"
                )

            reparam_substeps = 0

            ratio = (
                max_norm
                / float(self.neb.k[0])
            )

            reparam_converged = (
                ratio <= self.reparam_tol
            )

            while (
                not reparam_converged
                and reparam_substeps
                < self.max_reparam_substeps
            ):
                X = (
                    X
                    + self.reparam_alpha
                    * F_reparam
                )

                (
                    F_reparam,
                    max_norm,
                ) = (
                    self.reparam_force_function(
                        X
                    )
                )

                if not np.isfinite(max_norm):
                    raise FloatingPointError(
                        "Non-finite FENEB "
                        "reparameterization max_norm "
                        f"at outer step {self.nsteps}, "
                        f"substep {reparam_substeps}: "
                        f"{max_norm}"
                    )

                reparam_substeps += 1

                ratio = (
                    max_norm
                    / float(self.neb.k[0])
                )

                reparam_converged = (
                    ratio <= self.reparam_tol
                )

                if (
                    reparam_substeps == 1
                    or (
                        self.reparam_print_every > 0
                        and reparam_substeps
                        % self.reparam_print_every
                        == 0
                    )
                ):
                    print(
                        "[FENEB reparam] "
                        f"outer_step={self.nsteps}  "
                        f"substep="
                        f"{reparam_substeps}  "
                        f"reparam_alpha="
                        f"{self.reparam_alpha:.6e}  "
                        f"max_norm="
                        f"{max_norm:.6e}  "
                        f"max_norm/k="
                        f"{ratio:.6e}  "
                        f"target="
                        f"{self.reparam_tol:.6e}"
                    )

            # --------------------------------------------
            # 保存本次重参数化诊断
            # --------------------------------------------
            self.last_reparam_substeps = int(
                reparam_substeps
            )

            self.total_reparam_substeps += int(
                reparam_substeps
            )

            self.last_reparam_converged = bool(
                reparam_converged
            )

            self.last_reparam_max_norm = float(
                max_norm
            )

            self.last_reparam_ratio = float(
                ratio
            )

            # --------------------------------------------
            # 未达到准则时只警告，不停止
            # --------------------------------------------
            if not reparam_converged:
                self.reparam_limit_hits += 1

                print(
                    "\n[WARNING] FENEB "
                    "reparameterization reached "
                    "MAX_REPARAM_SUBSTEPS."
                )

                print(
                    "The current outer step will "
                    "continue instead of failing."
                )

                print(
                    f"  outer step        : "
                    f"{self.nsteps}"
                )

                print(
                    f"  physical alpha    : "
                    f"{self.alpha:.6e}"
                )

                print(
                    f"  reparam alpha     : "
                    f"{self.reparam_alpha:.6e}"
                )

                print(
                    f"  substeps          : "
                    f"{reparam_substeps}"
                )

                print(
                    f"  max_norm          : "
                    f"{max_norm:.12e}"
                )

                print(
                    f"  k                 : "
                    f"{self.neb.k[0]:.12e}"
                )

                print(
                    f"  max_norm/k        : "
                    f"{ratio:.12e}"
                )

                print(
                    f"  target ratio      : "
                    f"{self.reparam_tol:.12e}"
                )

                print(
                    f"  limit-hit count   : "
                    f"{self.reparam_limit_hits}"
                )

                print(
                    "  action             : "
                    "continue to next outer step\n"
                )

            # callback 会：
            #
            # log
            # call_observers
            # nsteps += 1
            self.callback(
                X
            )

        # ----------------------------------------------------
        # 完成 200 次更新后，对最终路径再评估一次
        #
        # 由此保存 count=200
        # ----------------------------------------------------
        F_final = (
            self.normalized_force_function(
                X
            )
        )

        self.record_current_state(
            physical_forces=F_final,
        )

        return bool(
            self.neb.get_residual()
            <= fmax
        )


# ============================================================
# 读取已有最终结果
# ============================================================
def load_existing_final_result(
    result_file,
    k_spring,
    alpha,
):
    with np.load(
        result_file,
        allow_pickle=True,
    ) as loaded:
        count_seq = np.asarray(
            loaded["count_seq"],
            dtype=float,
        )

        I_G_seq = np.asarray(
            loaded["I_perp_seq"],
            dtype=float,
        )

        I_E_seq = np.asarray(
            loaded[
                "I_perp_euclid_arc_seq"
            ],
            dtype=float,
        )

        I_arc_seq = np.asarray(
            loaded["I_arc_seq"],
            dtype=float,
        )

        Delta_F_seq = np.asarray(
            loaded["Delta_F_seq"],
            dtype=float,
        )

        fmax_seq = np.asarray(
            loaded["fmaxs"],
            dtype=float,
        )

    return {
        "k_spring": float(
            k_spring
        ),

        "alpha": float(
            alpha
        ),

        "skipped": True,
        "failed": False,

        "comparison_count": (
            float(count_seq[-1])
            if count_seq.size > 0
            else np.nan
        ),

        "final_I_perp_G": (
            float(I_G_seq[-1])
            if I_G_seq.size > 0
            else np.nan
        ),

        "final_I_perp_E": (
            float(I_E_seq[-1])
            if I_E_seq.size > 0
            else np.nan
        ),

        "final_I_arc": (
            float(I_arc_seq[-1])
            if I_arc_seq.size > 0
            else np.nan
        ),

        "final_Delta_F": (
            float(Delta_F_seq[-1])
            if Delta_F_seq.size > 0
            else np.nan
        ),

        "final_fmax": (
            float(fmax_seq[-1])
            if fmax_seq.size > 0
            else np.nan
        ),

        "result_file": result_file,
    }


# ============================================================
# 运行单组 k、alpha
# ============================================================
def run_feneb_experiment(
    k_spring,
    alpha,
):
    global GLOBAL_IMAGE_MD_CALLS

    GLOBAL_IMAGE_MD_CALLS = 0

    np.random.seed(seed)
    torch.manual_seed(seed)

    k_spring = float(
        k_spring
    )

    alpha = float(
        alpha
    )

    (
        casenum,
        folder_path,
        figure_path,
        result_file,
        done_file,
        failed_file,
    ) = feneb_result_paths(
        k_spring,
        alpha,
    )

    os.makedirs(
        folder_path,
        exist_ok=True,
    )

    os.makedirs(
        figure_path,
        exist_ok=True,
    )

    if FORCE_RERUN:
        if os.path.exists(done_file):
            os.remove(
                done_file
            )

        if os.path.exists(failed_file):
            os.remove(
                failed_file
            )

    if (
        os.path.exists(done_file)
        and os.path.exists(result_file)
        and not FORCE_RERUN
    ):
        print(
            f"[SKIP] k={k_spring:g}, "
            f"alpha={alpha:g} 已完成："
        )

        print(result_file)

        return load_existing_final_result(
            result_file,
            k_spring,
            alpha,
        )

    print("\n" + "=" * 110)
    print("ALANINE MFEP — FENEB GRID EXPERIMENT")
    print("=" * 110)
    print(
        f"FENEB import source     : "
        f"{FENEB_IMPORT_SOURCE}"
    )
    print(
        f"Internal images         : {N}"
    )
    print(
        f"Total images            : "
        f"{N_TOTAL_IMAGES}"
    )
    print(
        f"Spring constant k       : "
        f"{k_spring:g}"
    )
    print(
        f"Physical alpha          : "
        f"{alpha:g}"
    )
    print(
        f"Fixed reparam alpha     : "
        f"{REPARAM_ALPHA:g}"
    )
    print(
        f"Maximum count           : "
        f"{MAX_FENEB_STEPS}"
    )
    print(
        f"Physical fmax tolerance : "
        f"{FENEB_FMAX_TOL:g}"
    )
    print(
        f"Reparam tolerance       : "
        f"{REPARAM_TOL:g}"
    )
    print(
        f"Max reparam substeps    : "
        f"{MAX_REPARAM_SUBSTEPS}"
    )
    print(
        "Reparam limit behavior  : "
        "warning and continue"
    )
    print(
        "Initial path            : "
        "straight line"
    )
    print(
        f"Result folder           : "
        f"{folder_path}"
    )
    print("=" * 110)

    images = build_feneb_images()

    try:
        neb = NEB(
            images,
            k=k_spring,
            climb=False,
            parallel=False,
            method="feneb",
            allow_shared_calculator=False,
        )

    except Exception as exc:
        raise RuntimeError(
            "无法创建 method='feneb' 的 NEB。\n"
            "请确认当前导入的是包含 FENEB 的 "
            "自定义 neb.py。\n"
            f"Import source: "
            f"{FENEB_IMPORT_SOURCE}\n"
            f"Original error: {exc}"
        ) from exc

    neb.gradient_md_use = (
        feneb_gradient_md_use
    )

    # ========================================================
    # 保存序列
    # ========================================================
    count_seq = []
    optimizer_step_seq = []

    total_image_md_calls_seq = []
    first_image_force_calls_raw_seq = []
    first_image_force_calls_seq = []
    all_image_force_calls_seq = []
    path_gradient_sweeps_raw_seq = []

    images_seq = []

    M_ref_seq = []
    F_ref_seq = []
    Mg_ref_seq = []

    I_perp_seq = []
    I_perp_euclid_arc_seq = []
    I_arc_seq = []

    Delta_F_seq = []
    Delta_F_final_seq = []
    free_energy_profile_seq = []

    fmaxs = []
    fmax_feneb_physical_seq = []
    feneb_total_fmax_seq = []
    feneb_optimizer_residual_seq = []

    perpG_Mg_seq = []
    perpE_Mg_seq = []

    node_weights_G_seq = []
    node_weights_E_seq = []
    P_perp_G_seq = []

    arc_integrand_seq = []
    arc_speed_seq = []
    arc_inner_seq = []

    reparam_substeps_seq = []
    total_reparam_substeps_seq = []

    reparam_converged_seq = []
    reparam_max_norm_seq = []
    reparam_ratio_seq = []
    reparam_limit_hits_seq = []

    last_recorded_signature = {
        "step": None,
        "count": None,
    }

    start_time = time.perf_counter()

    # ========================================================
    # 保存当前 npz
    # ========================================================
    def save_current_npz():
        atomic_savez(
            result_file,

            # 横坐标和调用次数
            count_seq=np.asarray(
                count_seq,
                dtype=int,
            ),

            optimizer_step_seq=np.asarray(
                optimizer_step_seq,
                dtype=int,
            ),

            total_image_md_calls_seq=np.asarray(
                total_image_md_calls_seq,
                dtype=int,
            ),

            image_md_calls_seq=np.asarray(
                total_image_md_calls_seq,
                dtype=int,
            ),

            first_image_force_calls_raw_seq=np.asarray(
                first_image_force_calls_raw_seq,
                dtype=int,
            ),

            first_image_force_calls_seq=np.asarray(
                first_image_force_calls_seq,
                dtype=int,
            ),

            all_image_force_calls_seq=np.asarray(
                all_image_force_calls_seq,
                dtype=int,
            ),

            path_gradient_sweeps_raw_seq=np.asarray(
                path_gradient_sweeps_raw_seq,
                dtype=float,
            ),

            # 路径
            images_seq=np.asarray(
                images_seq,
                dtype=float,
            ),

            # Oracle
            M_ref_seq=np.asarray(
                M_ref_seq,
                dtype=float,
            ),

            F_ref_seq=np.asarray(
                F_ref_seq,
                dtype=float,
            ),

            Mg_ref_seq=np.asarray(
                Mg_ref_seq,
                dtype=float,
            ),

            # 四个指标
            I_perp_seq=np.asarray(
                I_perp_seq,
                dtype=float,
            ),

            I_perp_euclid_arc_seq=np.asarray(
                I_perp_euclid_arc_seq,
                dtype=float,
            ),

            I_arc_seq=np.asarray(
                I_arc_seq,
                dtype=float,
            ),

            Delta_F_seq=np.asarray(
                Delta_F_seq,
                dtype=float,
            ),

            Delta_F_final_seq=np.asarray(
                Delta_F_final_seq,
                dtype=float,
            ),

            free_energy_profile_seq=np.asarray(
                free_energy_profile_seq,
                dtype=float,
            ),

            # fmax
            fmaxs=np.asarray(
                fmaxs,
                dtype=float,
            ),

            fmax_feneb_physical_seq=np.asarray(
                fmax_feneb_physical_seq,
                dtype=float,
            ),

            feneb_total_fmax_seq=np.asarray(
                feneb_total_fmax_seq,
                dtype=float,
            ),

            feneb_optimizer_residual_seq=np.asarray(
                feneb_optimizer_residual_seq,
                dtype=float,
            ),

            # 投影数据
            perpG_Mg_seq=np.asarray(
                perpG_Mg_seq,
                dtype=float,
            ),

            perpE_Mg_seq=np.asarray(
                perpE_Mg_seq,
                dtype=float,
            ),

            node_weights_G_seq=np.asarray(
                node_weights_G_seq,
                dtype=float,
            ),

            node_weights_E_seq=np.asarray(
                node_weights_E_seq,
                dtype=float,
            ),

            P_perp_G_seq=np.asarray(
                P_perp_G_seq,
                dtype=float,
            ),

            # Arclength 数据
            arc_integrand_seq=np.asarray(
                arc_integrand_seq,
                dtype=float,
            ),

            arc_speed_seq=np.asarray(
                arc_speed_seq,
                dtype=float,
            ),

            arc_inner_seq=np.asarray(
                arc_inner_seq,
                dtype=float,
            ),

            # Reparameterization 诊断
            reparam_substeps_seq=np.asarray(
                reparam_substeps_seq,
                dtype=int,
            ),

            total_reparam_substeps_seq=np.asarray(
                total_reparam_substeps_seq,
                dtype=int,
            ),

            reparam_converged_seq=np.asarray(
                reparam_converged_seq,
                dtype=bool,
            ),

            reparam_max_norm_seq=np.asarray(
                reparam_max_norm_seq,
                dtype=float,
            ),

            reparam_ratio_seq=np.asarray(
                reparam_ratio_seq,
                dtype=float,
            ),

            reparam_limit_hits_seq=np.asarray(
                reparam_limit_hits_seq,
                dtype=int,
            ),

            # Metadata
            method=np.asarray(
                ["FENEB"],
                dtype=str,
            ),

            optimizer_method=np.asarray(
                ["feneb"],
                dtype=str,
            ),

            neb_method=np.asarray(
                ["feneb_central_tangent"],
                dtype=str,
            ),

            k_spring=np.asarray(
                [k_spring],
                dtype=float,
            ),

            alpha=np.asarray(
                [alpha],
                dtype=float,
            ),

            physical_alpha=np.asarray(
                [alpha],
                dtype=float,
            ),

            reparam_alpha=np.asarray(
                [REPARAM_ALPHA],
                dtype=float,
            ),

            reparam_tolerance=np.asarray(
                [REPARAM_TOL],
                dtype=float,
            ),

            max_reparam_substeps=np.asarray(
                [MAX_REPARAM_SUBSTEPS],
                dtype=int,
            ),

            reparam_limit_behavior=np.asarray(
                ["warning_and_continue"],
                dtype=str,
            ),

            fmax_tolerance=np.asarray(
                [FENEB_FMAX_TOL],
                dtype=float,
            ),

            max_steps=np.asarray(
                [MAX_FENEB_STEPS],
                dtype=int,
            ),

            max_count=np.asarray(
                [MAX_FENEB_STEPS],
                dtype=int,
            ),

            count_definition=np.asarray(
                [
                    "first interior image "
                    "M_gradient calls minus 1"
                ],
                dtype=str,
            ),

            N_internal=np.asarray(
                [N],
                dtype=int,
            ),

            N_total_images=np.asarray(
                [N_TOTAL_IMAGES],
                dtype=int,
            ),

            endpoints=np.stack(
                [C1, C2],
                axis=0,
            ),

            initial_path_type=np.asarray(
                ["straight_line"],
                dtype=str,
            ),

            grid_k=np.asarray(
                GRID_K,
                dtype=float,
            ),

            grid_alpha=np.asarray(
                GRID_ALPHA,
                dtype=float,
            ),
        )

    # ========================================================
    # 记录当前 state
    # ========================================================
    def record_state(
        optimizer,
        physical_forces,
    ):
        step_index = int(
            optimizer.nsteps
        )

        path_full = images_to_path(
            images
        )

        (
            M_all,
            F_all,
            Mg_all_cached,
        ) = collect_internal_oracle_data(
            images
        )

        metrics = compute_all_path_metrics(
            path_full,
            M_all,
            F_all,
        )

        (
            all_image_calls,
            first_image_calls_raw,
        ) = get_internal_image_md_calls(
            images
        )

        if not np.all(
            all_image_calls
            == first_image_calls_raw
        ):
            print(
                "[WARNING] Internal-image "
                "oracle call counts differ:"
            )
            print(all_image_calls)

        # 初始 oracle sweep 对应 count=0
        count_value = max(
            first_image_calls_raw - 1,
            0,
        )

        path_sweeps_raw = (
            GLOBAL_IMAGE_MD_CALLS
            / float(N)
        )

        if (
            last_recorded_signature["step"]
            == step_index
            and last_recorded_signature["count"]
            == count_value
        ):
            return

        physical_forces = np.asarray(
            physical_forces,
            dtype=float,
        )

        expected_size = (
            N * 2 * 3
        )

        if physical_forces.size != expected_size:
            raise ValueError(
                "Unexpected FENEB force size: "
                f"{physical_forces.size}; "
                f"expected {expected_size}."
            )

        physical_force_images = (
            physical_forces.reshape(
                N,
                2,
                3,
            )
        )

        feneb_total_fmax = float(
            np.max(
                np.linalg.norm(
                    physical_force_images.reshape(
                        -1,
                        3,
                    ),
                    axis=1,
                )
            )
        )

        try:
            optimizer_residual = float(
                optimizer.neb.get_residual()
            )

        except Exception:
            optimizer_residual = np.nan

        count_seq.append(
            int(count_value)
        )

        optimizer_step_seq.append(
            int(step_index)
        )

        total_image_md_calls_seq.append(
            int(GLOBAL_IMAGE_MD_CALLS)
        )

        first_image_force_calls_raw_seq.append(
            int(first_image_calls_raw)
        )

        first_image_force_calls_seq.append(
            int(count_value)
        )

        all_image_force_calls_seq.append(
            all_image_calls.copy()
        )

        path_gradient_sweeps_raw_seq.append(
            float(path_sweeps_raw)
        )

        images_seq.append(
            path_full.copy()
        )

        M_ref_seq.append(
            M_all.copy()
        )

        F_ref_seq.append(
            F_all.copy()
        )

        Mg_ref_seq.append(
            metrics["Mg_all"].copy()
        )

        I_perp_seq.append(
            metrics["I_perp_G"]
        )

        I_perp_euclid_arc_seq.append(
            metrics["I_perp_E"]
        )

        I_arc_seq.append(
            metrics["I_arc"]
        )

        Delta_F_seq.append(
            metrics["Delta_F_barrier"]
        )

        Delta_F_final_seq.append(
            metrics["Delta_F_final"]
        )

        free_energy_profile_seq.append(
            metrics[
                "free_energy_profile"
            ].copy()
        )

        fmaxs.append(
            metrics[
                "fmax_Gperp_Euclidean"
            ]
        )

        fmax_feneb_physical_seq.append(
            metrics[
                "fmax_FENEB_physical"
            ]
        )

        feneb_total_fmax_seq.append(
            feneb_total_fmax
        )

        feneb_optimizer_residual_seq.append(
            optimizer_residual
        )

        perpG_Mg_seq.append(
            metrics["perpG_Mg"].copy()
        )

        perpE_Mg_seq.append(
            metrics["perpE_Mg"].copy()
        )

        node_weights_G_seq.append(
            metrics[
                "node_weights_G"
            ].copy()
        )

        node_weights_E_seq.append(
            metrics[
                "node_weights_E"
            ].copy()
        )

        P_perp_G_seq.append(
            metrics[
                "P_perp_G_all"
            ].copy()
        )

        arc_integrand_seq.append(
            metrics[
                "arc_integrand"
            ].copy()
        )

        arc_speed_seq.append(
            metrics[
                "arc_speed"
            ].copy()
        )

        arc_inner_seq.append(
            metrics[
                "arc_inner"
            ].copy()
        )

        reparam_substeps_seq.append(
            int(
                optimizer
                .last_reparam_substeps
            )
        )

        total_reparam_substeps_seq.append(
            int(
                optimizer
                .total_reparam_substeps
            )
        )

        reparam_converged_seq.append(
            bool(
                optimizer
                .last_reparam_converged
            )
        )

        reparam_max_norm_seq.append(
            float(
                optimizer
                .last_reparam_max_norm
            )
        )

        reparam_ratio_seq.append(
            float(
                optimizer
                .last_reparam_ratio
            )
        )

        reparam_limit_hits_seq.append(
            int(
                optimizer
                .reparam_limit_hits
            )
        )

        last_recorded_signature["step"] = (
            step_index
        )

        last_recorded_signature["count"] = (
            count_value
        )

        elapsed = (
            time.perf_counter()
            - start_time
        )

        print(
            f"[FENEB "
            f"k={k_spring:g} "
            f"alpha={alpha:g}] "
            f"step={step_index:4d}  "
            f"count={count_value:4d}  "
            f"I_G="
            f"{metrics['I_perp_G']:.6e}  "
            f"I_E="
            f"{metrics['I_perp_E']:.6e}  "
            f"I_arc="
            f"{metrics['I_arc']:.6e}  "
            f"DeltaF="
            f"{metrics['Delta_F_barrier']:.6f}  "
            f"fmax_Gperp_E="
            f"{metrics['fmax_Gperp_Euclidean']:.6e}  "
            f"FENEB_total_fmax="
            f"{feneb_total_fmax:.6e}  "
            f"optimizer_residual="
            f"{optimizer_residual:.6e}  "
            f"previous_reparam_ok="
            f"{optimizer.last_reparam_converged}  "
            f"reparam_limit_hits="
            f"{optimizer.reparam_limit_hits}  "
            f"total_MD_calls="
            f"{GLOBAL_IMAGE_MD_CALLS}  "
            f"elapsed={elapsed:.1f}s"
        )

        if (
            SAVE_FIGS
            and (
                count_value == 0
                or count_value
                % SAVE_FIG_EVERY == 0
                or count_value
                == MAX_FENEB_STEPS
            )
        ):
            try:
                fig_save(
                    count_value,

                    path_full[1:-1],
                    path_full[1:-1],
                    None,
                    path_full[1:-1],

                    xlim=xlim,
                    ylim=ylim,

                    force1=-metrics[
                        "perpG_Mg"
                    ],

                    force2=-F_all,

                    force3=-metrics[
                        "perpE_Mg"
                    ],

                    figure_path=figure_path,
                )

            except Exception as exc:
                print(
                    "[WARNING] fig_save failed "
                    f"at count={count_value}: "
                    f"{exc}"
                )

            plt.close("all")

        if SAVE_EVERY_STATE:
            save_current_npz()

    # ========================================================
    # 创建 optimizer
    # ========================================================
    optimizer = RecordingFENEBOptimizer(
        neb,

        method="feneb",

        # Grid 调节的 physical alpha
        alpha=alpha,

        # 固定的 reparameterization alpha
        reparam_alpha=REPARAM_ALPHA,

        reparam_tol=REPARAM_TOL,

        max_reparam_substeps=(
            MAX_REPARAM_SUBSTEPS
        ),

        reparam_print_every=(
            REPARAM_PRINT_EVERY
        ),

        logfile=None,
        trajectory=None,
        verbose=0,

        state_recorder=record_state,
    )

    # ========================================================
    # 执行 FENEB
    # ========================================================
    converged = optimizer.run(
        fmax=FENEB_FMAX_TOL,
        steps=MAX_FENEB_STEPS,
    )

    save_current_npz()

    elapsed_total = (
        time.perf_counter()
        - start_time
    )

    (
        final_all_image_calls,
        final_first_image_calls,
    ) = get_internal_image_md_calls(
        images
    )

    final_count = max(
        final_first_image_calls - 1,
        0,
    )

    final_I_G = (
        float(I_perp_seq[-1])
        if len(I_perp_seq) > 0
        else np.nan
    )

    final_I_E = (
        float(
            I_perp_euclid_arc_seq[-1]
        )
        if len(
            I_perp_euclid_arc_seq
        ) > 0
        else np.nan
    )

    final_I_arc = (
        float(I_arc_seq[-1])
        if len(I_arc_seq) > 0
        else np.nan
    )

    final_Delta_F = (
        float(Delta_F_seq[-1])
        if len(Delta_F_seq) > 0
        else np.nan
    )

    final_Delta_F_end = (
        float(Delta_F_final_seq[-1])
        if len(
            Delta_F_final_seq
        ) > 0
        else np.nan
    )

    final_fmax = (
        float(fmaxs[-1])
        if len(fmaxs) > 0
        else np.nan
    )

    # ========================================================
    # 写 DONE
    # ========================================================
    with open(
        done_file,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            "completed\n"
            f"method=FENEB\n"
            f"converged={converged}\n"
            f"k={k_spring}\n"
            f"physical_alpha={alpha}\n"
            f"reparam_alpha="
            f"{REPARAM_ALPHA}\n"
            f"reparam_tolerance="
            f"{REPARAM_TOL}\n"
            f"max_reparam_substeps="
            f"{MAX_REPARAM_SUBSTEPS}\n"
            f"reparam_limit_behavior="
            f"warning_and_continue\n"
            f"reparam_limit_hits="
            f"{optimizer.reparam_limit_hits}\n"
            f"fmax_tolerance="
            f"{FENEB_FMAX_TOL}\n"
            f"max_count="
            f"{MAX_FENEB_STEPS}\n"
            f"optimizer_steps="
            f"{optimizer.nsteps}\n"
            f"comparison_count="
            f"{final_count}\n"
            f"first_image_md_calls="
            f"{final_first_image_calls}\n"
            f"total_image_md_calls="
            f"{GLOBAL_IMAGE_MD_CALLS}\n"
            f"all_image_md_calls="
            f"{final_all_image_calls.tolist()}\n"
            f"total_reparam_substeps="
            f"{optimizer.total_reparam_substeps}\n"
            f"final_reparam_converged="
            f"{optimizer.last_reparam_converged}\n"
            f"final_reparam_ratio="
            f"{optimizer.last_reparam_ratio}\n"
            f"final_I_perp_G="
            f"{final_I_G}\n"
            f"final_I_perp_E="
            f"{final_I_E}\n"
            f"final_I_arc="
            f"{final_I_arc}\n"
            f"final_Delta_F="
            f"{final_Delta_F}\n"
            f"final_Delta_F_end="
            f"{final_Delta_F_end}\n"
            f"final_fmax="
            f"{final_fmax}\n"
            f"elapsed_seconds="
            f"{elapsed_total}\n"
            f"result_file="
            f"{result_file}\n"
        )

    print("\n" + "=" * 110)
    print("FENEB EXPERIMENT FINISHED")
    print("=" * 110)
    print(
        f"k                       : "
        f"{k_spring:g}"
    )
    print(
        f"Physical alpha          : "
        f"{alpha:g}"
    )
    print(
        f"Reparam alpha           : "
        f"{REPARAM_ALPHA:g}"
    )
    print(
        f"Converged               : "
        f"{converged}"
    )
    print(
        f"Optimizer steps         : "
        f"{optimizer.nsteps}"
    )
    print(
        f"Final count             : "
        f"{final_count}"
    )
    print(
        f"First-image MD calls    : "
        f"{final_first_image_calls}"
    )
    print(
        f"Total image MD calls    : "
        f"{GLOBAL_IMAGE_MD_CALLS}"
    )
    print(
        f"Total reparam substeps  : "
        f"{optimizer.total_reparam_substeps}"
    )
    print(
        f"Reparam limit hits      : "
        f"{optimizer.reparam_limit_hits}"
    )
    print(
        f"Final reparam converged : "
        f"{optimizer.last_reparam_converged}"
    )
    print(
        f"Final reparam ratio     : "
        f"{optimizer.last_reparam_ratio:.6e}"
    )
    print(
        f"Final I_perp_G          : "
        f"{final_I_G:.6e}"
    )
    print(
        f"Final I_perp_E          : "
        f"{final_I_E:.6e}"
    )
    print(
        f"Final I_arc             : "
        f"{final_I_arc:.6e}"
    )
    print(
        f"Final Delta F barrier   : "
        f"{final_Delta_F:.6f}"
    )
    print(
        f"Final Delta F endpoint  : "
        f"{final_Delta_F_end:.6f}"
    )
    print(
        f"Final fmax              : "
        f"{final_fmax:.6e}"
    )
    print(
        f"Elapsed seconds         : "
        f"{elapsed_total:.1f}"
    )
    print(
        f"Result file             : "
        f"{result_file}"
    )
    print(
        f"Figure folder           : "
        f"{figure_path}"
    )
    print("=" * 110)

    return {
        "k_spring": k_spring,
        "alpha": alpha,

        "physical_alpha": alpha,
        "reparam_alpha": REPARAM_ALPHA,

        "skipped": False,
        "failed": False,

        "converged": bool(
            converged
        ),

        "optimizer_steps": int(
            optimizer.nsteps
        ),

        "comparison_count": int(
            final_count
        ),

        "first_image_md_calls": int(
            final_first_image_calls
        ),

        "total_image_md_calls": int(
            GLOBAL_IMAGE_MD_CALLS
        ),

        "total_reparam_substeps": int(
            optimizer.total_reparam_substeps
        ),

        "reparam_limit_hits": int(
            optimizer.reparam_limit_hits
        ),

        "final_reparam_converged": bool(
            optimizer.last_reparam_converged
        ),

        "final_reparam_ratio": float(
            optimizer.last_reparam_ratio
        ),

        "final_I_perp_G": final_I_G,
        "final_I_perp_E": final_I_E,
        "final_I_arc": final_I_arc,

        "final_Delta_F": (
            final_Delta_F
        ),

        "final_Delta_F_end": (
            final_Delta_F_end
        ),

        "final_fmax": final_fmax,

        "elapsed_seconds": float(
            elapsed_total
        ),

        "result_file": result_file,
        "folder_path": folder_path,
        "done_file": done_file,
    }


# ============================================================
# 保存 grid 汇总
# ============================================================
def save_grid_summary(
    sweep_results,
):
    summary_file = os.path.join(
        base_path,
        (
            "MFEP_FENEB_GRID_SWEEP_"
            f"N{N}"
            f"_MAXCOUNT{MAX_FENEB_STEPS}"
            f"_REPARAMALPHA"
            f"{REPARAM_ALPHA:g}"
            f"_SUMMARY.npz"
        ),
    )

    atomic_savez(
        summary_file,

        results=np.array(
            sweep_results,
            dtype=object,
        ),

        grid_k=np.asarray(
            GRID_K,
            dtype=float,
        ),

        grid_alpha=np.asarray(
            GRID_ALPHA,
            dtype=float,
        ),

        reparam_alpha=np.asarray(
            [REPARAM_ALPHA],
            dtype=float,
        ),

        reparam_tolerance=np.asarray(
            [REPARAM_TOL],
            dtype=float,
        ),

        max_reparam_substeps=np.asarray(
            [MAX_REPARAM_SUBSTEPS],
            dtype=int,
        ),

        reparam_limit_behavior=np.asarray(
            ["warning_and_continue"],
            dtype=str,
        ),

        max_feneb_steps=np.asarray(
            [MAX_FENEB_STEPS],
            dtype=int,
        ),

        max_count=np.asarray(
            [MAX_FENEB_STEPS],
            dtype=int,
        ),

        fmax_tolerance=np.asarray(
            [FENEB_FMAX_TOL],
            dtype=float,
        ),

        N_internal=np.asarray(
            [N],
            dtype=int,
        ),

        endpoints=np.stack(
            [C1, C2],
            axis=0,
        ),
    )

    return summary_file


# ============================================================
# 主程序
# ============================================================
def main():
    print("=" * 120)
    print(
        "ALANINE MFEP — FENEB K/ALPHA GRID SWEEP"
    )
    print("=" * 120)
    print(
        f"FENEB import source     : "
        f"{FENEB_IMPORT_SOURCE}"
    )
    print(
        f"Spring constants        : "
        f"{GRID_K}"
    )
    print(
        f"Physical alpha grid     : "
        f"{GRID_ALPHA}"
    )
    print(
        f"Fixed reparam alpha     : "
        f"{REPARAM_ALPHA}"
    )
    print(
        f"Reparam tolerance       : "
        f"{REPARAM_TOL}"
    )
    print(
        f"Max reparam substeps    : "
        f"{MAX_REPARAM_SUBSTEPS}"
    )
    print(
        "Reparam limit behavior  : "
        "warning and continue"
    )
    print(
        f"Total experiments       : "
        f"{len(GRID_K) * len(GRID_ALPHA)}"
    )
    print(
        f"N internal images       : {N}"
    )
    print(
        f"Maximum count           : "
        f"{MAX_FENEB_STEPS}"
    )
    print(
        f"Physical fmax tolerance : "
        f"{FENEB_FMAX_TOL}"
    )
    print(
        f"Force rerun             : "
        f"{FORCE_RERUN}"
    )
    print(
        f"Base path               : "
        f"{base_path}"
    )
    print("=" * 120)

    sweep_results = []

    configurations = [
        (
            float(k_spring),
            float(alpha),
        )
        for k_spring in GRID_K
        for alpha in GRID_ALPHA
    ]

    total_experiments = len(
        configurations
    )

    for experiment_index, (
        k_spring,
        alpha,
    ) in enumerate(
        configurations,
        start=1,
    ):
        print("\n" + "#" * 120)
        print(
            f"RUNNING FENEB EXPERIMENT "
            f"{experiment_index}/"
            f"{total_experiments}"
        )
        print(
            f"k={k_spring:g}, "
            f"physical alpha={alpha:g}, "
            f"reparam alpha="
            f"{REPARAM_ALPHA:g}"
        )
        print("#" * 120)

        (
            casenum,
            folder_path,
            figure_path,
            result_file,
            done_file,
            failed_file,
        ) = feneb_result_paths(
            k_spring,
            alpha,
        )

        try:
            result = run_feneb_experiment(
                k_spring,
                alpha,
            )

        except KeyboardInterrupt:
            print(
                "\nFENEB grid sweep "
                "manually interrupted."
            )

            summary_file = save_grid_summary(
                sweep_results
            )

            print(
                "\nPartial summary saved to:"
            )
            print(summary_file)

            raise

        except Exception as exc:
            # 这里只处理真正的异常，例如：
            #
            # NaN
            # 文件错误
            # oracle 错误
            # shape 错误
            #
            # 重参数化达到上限不会进入这里。
            print(
                f"\n[ERROR] "
                f"k={k_spring:g}, "
                f"alpha={alpha:g} failed."
            )

            print(
                f"  type   : "
                f"{type(exc).__name__}"
            )

            print(
                f"  reason : {exc}"
            )

            print("\nFull traceback:")
            traceback.print_exc()

            os.makedirs(
                folder_path,
                exist_ok=True,
            )

            with open(
                failed_file,
                "w",
                encoding="utf-8",
            ) as file:
                file.write(
                    "failed\n"
                    f"k={k_spring}\n"
                    f"physical_alpha={alpha}\n"
                    f"reparam_alpha="
                    f"{REPARAM_ALPHA}\n"
                    f"error_type="
                    f"{type(exc).__name__}\n"
                    f"error_message="
                    f"{str(exc)}\n"
                )

            result = {
                "k_spring": float(
                    k_spring
                ),

                "alpha": float(
                    alpha
                ),

                "physical_alpha": float(
                    alpha
                ),

                "reparam_alpha": float(
                    REPARAM_ALPHA
                ),

                "skipped": False,
                "failed": True,

                "error_type": (
                    type(exc).__name__
                ),

                "error_message": str(
                    exc
                ),

                "result_file": result_file,
                "folder_path": folder_path,
                "failed_file": failed_file,
            }

        sweep_results.append(
            result
        )

        summary_file = save_grid_summary(
            sweep_results
        )

        print(
            "\nCurrent grid summary saved to:"
        )
        print(summary_file)

    # ========================================================
    # 最终汇总
    # ========================================================
    print("\n" + "=" * 165)
    print(
        "FENEB GRID SWEEP FINAL SUMMARY"
    )
    print("=" * 165)

    successful_results = [
        result
        for result in sweep_results
        if not result.get(
            "failed",
            False,
        )
    ]

    failed_results = [
        result
        for result in sweep_results
        if result.get(
            "failed",
            False,
        )
    ]

    successful_sorted = sorted(
        successful_results,
        key=lambda result: (
            result.get(
                "final_I_perp_G",
                np.inf,
            )
        ),
    )

    for rank, result in enumerate(
        successful_sorted,
        start=1,
    ):
        print(
            f"rank={rank:2d}"
            f" | k="
            f"{result.get('k_spring', np.nan):>8g}"
            f" | physical_alpha="
            f"{result.get('alpha', np.nan):>10g}"
            f" | reparam_alpha="
            f"{result.get('reparam_alpha', np.nan):>10g}"
            f" | converged="
            f"{result.get('converged', False)}"
            f" | count="
            f"{result.get('comparison_count', np.nan):>6}"
            f" | reparam_limit_hits="
            f"{result.get('reparam_limit_hits', 0):>5}"
            f" | I_G="
            f"{result.get('final_I_perp_G', np.nan):.6e}"
            f" | I_E="
            f"{result.get('final_I_perp_E', np.nan):.6e}"
            f" | I_arc="
            f"{result.get('final_I_arc', np.nan):.6e}"
            f" | DeltaF="
            f"{result.get('final_Delta_F', np.nan):.6f}"
            f" | fmax="
            f"{result.get('final_fmax', np.nan):.6e}"
            f" | elapsed="
            f"{result.get('elapsed_seconds', np.nan):.1f}s"
        )

    if len(failed_results) > 0:
        print("\n" + "-" * 165)
        print("FAILED CONFIGURATIONS")
        print("-" * 165)

        for result in failed_results:
            print(
                f"k="
                f"{result.get('k_spring', np.nan):g}"
                f" | physical alpha="
                f"{result.get('alpha', np.nan):g}"
                f" | "
                f"{result.get('error_type', '')}: "
                f"{result.get('error_message', '')}"
            )

    print("=" * 165)

    summary_file = save_grid_summary(
        sweep_results
    )

    print("\nFinal grid summary:")
    print(summary_file)

    if len(successful_sorted) > 0:
        best = successful_sorted[0]

        print(
            "\nBest configuration by "
            "final I_perp_G:"
        )

        print(
            f"  k                  = "
            f"{best['k_spring']:g}"
        )

        print(
            f"  physical alpha     = "
            f"{best['alpha']:g}"
        )

        print(
            f"  reparam alpha      = "
            f"{best['reparam_alpha']:g}"
        )

        print(
            f"  count              = "
            f"{best.get('comparison_count', np.nan)}"
        )

        print(
            f"  reparam limit hits = "
            f"{best.get('reparam_limit_hits', 0)}"
        )

        print(
            f"  I_G                = "
            f"{best.get('final_I_perp_G', np.nan):.6e}"
        )

        print(
            f"  I_E                = "
            f"{best.get('final_I_perp_E', np.nan):.6e}"
        )

        print(
            f"  I_arc              = "
            f"{best.get('final_I_arc', np.nan):.6e}"
        )

        print(
            f"  Delta F            = "
            f"{best.get('final_Delta_F', np.nan):.6f}"
        )

        print(
            f"  result             = "
            f"{best.get('result_file', '')}"
        )


if __name__ == "__main__":
    main()

ALANINE MFEP — FENEB K/ALPHA GRID SWEEP
FENEB import source     : ase.mep.neb
Spring constants        : [1.0]
Physical alpha grid     : [0.0001, 1e-05, 1e-06, 1e-07]
Fixed reparam alpha     : 0.001
Reparam tolerance       : 0.001
Max reparam substeps    : 100000
Reparam limit behavior  : warning and continue
Total experiments       : 4
N internal images       : 20
Maximum count           : 200
Physical fmax tolerance : 0.01
Force rerun             : True
Base path               : /Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo

########################################################################################################################
RUNNING FENEB EXPERIMENT 1/4
k=1, physical alpha=0.0001, reparam alpha=0.001
########################################################################################################################

ALANINE MFEP — FENEB GRID EXPERIMENT
FENEB import source     : ase.mep.neb
Internal images         : 20
Total images            : 22
Spring cons

KeyboardInterrupt: 